# TensorRT 기초: PyTorch → ONNX → TensorRT

> 실행 환경: Colab GPU 런타임 (VS Code: **Select Kernel → Colab → GPU**)
> study와 무관한 boilerplate(캐시, 추론 래퍼, 벤치마크)는 [`src/vopt`](../../src/vopt)에 있습니다.

## 0. Setup
- Colab에 기본 설치된 torch는 그대로 사용하고, 없는 패키지만 `uv`로 설치합니다.
- 이 저장소를 clone해서 `vopt` 헬퍼를 import합니다. 로컬에서 `src/vopt`를 수정했다면 push한 뒤 이 셀을 다시 실행하세요(`git pull` + autoreload).

In [1]:
# 0. Setup (Colab 런타임에서 실행)
import os, subprocess, sys

REPO_URL = "https://github.com/limJhyeok/vopt-learning.git"
REPO_DIR = "/content/vopt-learning"
PKGS = ["tensorrt-cu12", "onnx", "onnxscript"]  # Colab 기본 torch/torchvision은 재설치하지 않음

assert os.path.isdir("/content"), "Colab 런타임이 아닙니다. 커널을 Colab으로 선택하세요."

def sh(cmd: str) -> None:
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

sh("pip install -q uv")
if os.path.isdir(REPO_DIR):
    sh(f"git -C {REPO_DIR} pull -q --ff-only")
else:
    sh(f"git clone -q --depth 1 {REPO_URL} {REPO_DIR}")
sh("uv pip install --system -q " + " ".join(PKGS))

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# autoreload: src/vopt 수정 사항을 커널 재시작 없이 반영
# Colab의 구버전 IPython autoreload는 Python 3.12+에서 삭제된 `imp`를 import하므로 최소한의 shim을 넣는다.
import importlib, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401  (Python <= 3.11)
    except ModuleNotFoundError:
        sys.modules["imp"] = types.SimpleNamespace(reload=importlib.reload)
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as e:
    print(f"[warn] autoreload 비활성화 ({e!r}). vopt 수정 후에는 커널을 재시작하세요.")

from vopt.env import print_env
print_env()

$ pip install -q uv
$ git clone -q --depth 1 https://github.com/limJhyeok/vopt-learning.git /content/vopt-learning
$ uv pip install --system -q tensorrt-cu12 onnx onnxscript
python       : 3.13.16
torch        : 2.14.1+cu130
cuda (torch) : 13.0
gpu          : Tesla T4
gpu tag      : tesla-t4-sm75
tensorrt     : 11.4.0.106


> **(선택) Google Drive 캐시**: 명령 팔레트 → `Colab: Mount Google Drive to Server...`로 Drive를 마운트하면,
> 빌드한 engine이 `MyDrive/vopt-cache`에 저장되어 런타임이 바뀌어도 유지됩니다. (마운트하지 않으면 `/content/vopt-cache`)

## 1. PyTorch 모델 → ONNX

In [2]:
import copy
import torch
import torchvision
from vopt import cache, data

B = 8
model = torchvision.models.resnet18(weights="DEFAULT").eval().cuda()
# 실제 사진 B장(Oxford-IIIT Pet, CC BY-SA 4.0. HF에서 필요한 장수만 받음)을 ResNet 표준 전처리한 [B, 3, 224, 224] 배치.
# 랜덤 노이즈는 logit 1·2위 차이가 작아 정밀도 비교(top-1 일치율)가 부정확하다.
x = data.imagenet_batch(B)

# 최신 TensorRT는 strongly typed: engine 정밀도 = ONNX의 dtype.
# ONNX/engine 입력 shape은 export할 때의 x shape으로 고정되므로 파일명에 B를 넣는다.
onnx_path = cache.cache_dir() / f"resnet18_b{B}.onnx"
if not onnx_path.exists():
    torch.onnx.export(
        model, (x,), str(onnx_path),
        input_names=["input"], output_names=["logits"],
        external_data=False,  # weight를 .onnx 하나에 저장 (True면 .onnx.data로 분리되어 parser.parse(bytes)가 못 찾음)
        dynamo=True,  # torch.export 기반 exporter (onnxscript 필요)
    )
print(onnx_path, f"{onnx_path.stat().st_size / 1e6:.1f} MB")

# → FP16 engine을 원하면 FP16 모델 자체를 export한다.
model_fp16 = copy.deepcopy(model).half()
x_fp16 = x.half()

fp16_onnx_path = cache.cache_dir() / f"resnet18_fp16_b{B}.onnx"
if not fp16_onnx_path.exists():
    torch.onnx.export(
        model_fp16, (x_fp16,), str(fp16_onnx_path),
        input_names=["input"], output_names=["logits"],
        external_data=False,  # weight를 .onnx 하나에 저장 (True면 .onnx.data로 분리되어 parser.parse(bytes)가 못 찾음)
        dynamo=True,  # torch.export 기반 exporter (onnxscript 필요)
    )
print(fp16_onnx_path, f"{fp16_onnx_path.stat().st_size / 1e6:.1f} MB")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s]


[data] timm/oxford-iiit-pet (test) 이미지 8장 다운로드 중...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
/content/vopt-cache/resnet18_b8.onnx 46.8 MB
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
/co

## 2. ONNX → TensorRT engine 빌드
TensorRT Python API로 직접 변환합니다. 큰 흐름은 항상 같습니다.

1. **Logger**: TensorRT가 경고/에러를 출력할 곳
2. **Builder**: engine을 만드는 주체
3. **Network**: 빈 네트워크 정의를 만들고, **OnnxParser**로 ONNX 그래프를 채워 넣음
4. **BuilderConfig**: 빌드 옵션(workspace 메모리 한도 등)
5. **build_serialized_network**: GPU에서 레이어별 최적 kernel을 탐색(수십 초~수 분) → 직렬화된 engine(bytes)
6. bytes를 `.engine` 파일로 저장

> TensorRT 11.x 기준입니다.

**정밀도**: TensorRT 11은 *strongly typed* 네트워크만 지원합니다. 즉 engine 정밀도 = ONNX의 dtype입니다.
FP16으로 빌드하라는 옵션(`BuilderFlag.FP16`)이 없으므로, FP16 engine은 1장에서 export한 FP16 ONNX로 빌드합니다.

In [3]:
import tensorrt as trt
from vopt import env

def build_engine(onnx_path, engine_path):
    # 1. Logger
    logger = trt.Logger(trt.Logger.WARNING)  # 빌드 과정을 자세히 보려면 trt.Logger.VERBOSE

    # 2. Builder
    builder = trt.Builder(logger)

    # 3. Network + ONNX Parser
    #    TRT 11은 strongly typed가 기본: 각 레이어 정밀도 = ONNX의 dtype (별도 플래그 불필요)
    network = builder.create_network()
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX 파싱 실패")

    for i in range(network.num_inputs):
        t = network.get_input(i)
        print(f"  input : {t.name} {tuple(t.shape)} {t.dtype}")
    for i in range(network.num_outputs):
        t = network.get_output(i)
        print(f"  output: {t.name} {tuple(t.shape)} {t.dtype}")

    # 4. Builder config
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # kernel이 쓸 임시 메모리 한도 1GB

    # 5. 빌드: 현재 GPU에서 레이어별로 가장 빠른 kernel을 고른다
    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        raise RuntimeError("engine 빌드 실패 (Logger를 VERBOSE로 바꿔 로그 확인)")

    # 6. 저장
    with open(engine_path, "wb") as f:
        f.write(engine_bytes)
    print(f"  saved : {engine_path} ({engine_path.stat().st_size / 1e6:.1f} MB)")

In [4]:
# ⚠️ 입력 shape 고정: ONNX를 x의 shape [B, 3, 224, 224]로 export했으므로 engine도 이 shape 전용이다.
#    batch 크기나 해상도가 다른 입력을 넣으면 실행 시 에러가 난다. (여러 shape 지원 = dynamic shape은 다음 노트북에서)
# engine은 빌드한 GPU 아키텍처와 TensorRT 버전에서만 동작하므로 파일명에 둘 다 넣는다.
# 이미 빌드한 engine이 있으면 재사용한다(빌드 과정을 다시 보고 싶으면 파일을 지우고 실행).
suffix = f"b{B}_{env.gpu_tag()}_trt{trt.__version__}.engine"
engine_path = cache.cache_dir() / f"resnet18_{suffix}"
fp16_engine_path = cache.cache_dir() / f"resnet18_fp16_{suffix}"

for src_path, dst_path in [(onnx_path, engine_path), (fp16_onnx_path, fp16_engine_path)]:
    print(f"{src_path.name} → {dst_path.name}")
    if dst_path.exists():
        print("  이미 있음, 빌드 생략")
    else:
        build_engine(src_path, dst_path)

resnet18_b8.onnx → resnet18_b8_tesla-t4-sm75_trt11.4.0.106.engine
  input : input (8, 3, 224, 224) DataType.FLOAT
  output: logits (8, 1000) DataType.FLOAT
  saved : /content/vopt-cache/resnet18_b8_tesla-t4-sm75_trt11.4.0.106.engine (54.2 MB)
resnet18_fp16_b8.onnx → resnet18_fp16_b8_tesla-t4-sm75_trt11.4.0.106.engine
  input : input (8, 3, 224, 224) DataType.HALF
  output: logits (8, 1000) DataType.HALF
  saved : /content/vopt-cache/resnet18_fp16_b8_tesla-t4-sm75_trt11.4.0.106.engine (23.7 MB)


### engine 로드 → 추론
빌드한 `.engine` 파일로 추론하는 흐름입니다.

1. **Runtime**으로 engine bytes를 역직렬화 → `ICudaEngine`
2. engine에서 **ExecutionContext** 생성 (추론 1회분의 상태: 입출력 주소 등)
3. 입력/출력 GPU 메모리를 준비하고 `set_tensor_address`로 **이름 ↔ 주소**를 연결
4. `execute_async_v3(stream)`으로 실행

GPU 메모리는 torch 텐서를 그대로 씁니다(`tensor.data_ptr()` = GPU 주소). 그래서 별도의 CUDA 메모리 할당/복사 코드가 필요 없습니다.

In [5]:
# 1. Runtime으로 역직렬화
logger = trt.Logger(trt.Logger.WARNING)
runtime = trt.Runtime(logger)

with open(engine_path, "rb") as f:
    engine = runtime.deserialize_cuda_engine(f.read())
assert engine is not None, "역직렬화 실패: 다른 GPU 아키텍처/TRT 버전에서 빌드된 engine일 수 있음"

# engine의 입출력 텐서 확인: 이름은 ONNX export 때 준 input_names/output_names
for i in range(engine.num_io_tensors):
    name = engine.get_tensor_name(i)
    print(f"{engine.get_tensor_mode(name)}  {name}  {engine.get_tensor_shape(name)}  {engine.get_tensor_dtype(name)}")

# 2. ExecutionContext
context = engine.create_execution_context()

TensorIOMode.INPUT  input  (8, 3, 224, 224)  DataType.FLOAT
TensorIOMode.OUTPUT  logits  (8, 1000)  DataType.FLOAT


In [6]:
# 3~4. 추론 함수
def trt_infer(context, x):
    x = x.contiguous()  # TensorRT는 연속된 메모리를 가정

    # 출력 버퍼: engine이 알려주는 shape으로 미리 할당 (이 모델은 출력 dtype = 입력 dtype)
    out_shape = tuple(context.get_tensor_shape("logits"))
    out = torch.empty(out_shape, dtype=x.dtype, device="cuda")

    # 텐서 이름 ↔ GPU 주소 연결
    # (shape 고정 engine이라 set_input_shape이 필요 없다. x의 shape이 build 때와 다르면 실행이 실패한다.)
    context.set_tensor_address("input", x.data_ptr())
    context.set_tensor_address("logits", out.data_ptr())

    # torch와 같은 CUDA stream에서 실행 → 이후 torch 연산이 자동으로 실행 완료를 기다린다
    ok = context.execute_async_v3(torch.cuda.current_stream().cuda_stream)
    assert ok, "TensorRT 실행 실패"
    return out

In [7]:
# PyTorch 출력과 비교해 변환이 맞는지 확인
with torch.inference_mode():
    y_torch = model(x)
y_trt = trt_infer(context, x)
print("max |diff|  :", (y_trt - y_torch).abs().max().item())
print("top-1 일치 :", (y_trt.argmax(-1) == y_torch.argmax(-1)).float().mean().item() * 100, "%")

# FP16 engine도 같은 방법으로 로드
with open(fp16_engine_path, "rb") as f:
    fp16_engine = runtime.deserialize_cuda_engine(f.read())
fp16_context = fp16_engine.create_execution_context()

# 아래 벤치마크에서 model(x)처럼 호출할 수 있게 감싼다
# (engine 변수는 context가 쓰는 동안 살아 있어야 하므로 지우지 않는다)
trt_model = lambda inp: trt_infer(context, inp)
fp16_trt_model = lambda inp: trt_infer(fp16_context, inp)

max |diff|  : 1.811981201171875e-05
top-1 일치 : 100.0 %


## 3. 정확도 & 속도 비교

In [8]:
model_autocast = copy.deepcopy(model)

def run_autocast(inp):
  with torch.amp.autocast("cuda", dtype=torch.float16):
    return model_autocast(inp)

model_compiled = torch.compile(copy.deepcopy(model))
model_compiled_fp16 = torch.compile(copy.deepcopy(model_fp16))

In [9]:
from vopt.bench import benchmark, compare

compare({
    "pytorch fp32": benchmark(lambda: model(x)),
    "pytorch fp16": benchmark(lambda: model_fp16(x_fp16)),
    "pytorch autocast": benchmark(lambda: run_autocast(x)),
    "pytorch compiled fp32": benchmark(lambda: model_compiled(x)),
    "pytorch compiled fp16": benchmark(lambda: model_compiled_fp16(x_fp16)),
    "tensorrt fp32": benchmark(lambda: trt_model(x)),
    "tensorrt fp16": benchmark(lambda: fp16_trt_model(x_fp16)),
})

W1115 11:58:58.099000 402 torch/_inductor/utils.py:1806] [0/0] Not enough SMs to use max_autotune_gemm mode


name                      mean(ms)       p50       p90   speedup   max_diff    rel_err   cos_sim     top1
---------------------------------------------------------------------------------------------------------
pytorch fp32                 8.950     8.987     9.141     1.00x     0.0000   0.00e+00  1.000000   100.0%
pytorch fp16                 4.047     3.974     4.141     2.21x     0.0244   1.81e-03  0.999997   100.0%
pytorch autocast             4.094     4.081     4.154     2.19x     0.0176   9.97e-04  0.999999   100.0%
pytorch compiled fp32        8.798     8.789     8.923     1.02x     0.0000   1.03e-06  1.000000   100.0%
pytorch compiled fp16        2.974     2.418     4.246     3.01x     0.0254   1.78e-03  0.999998   100.0%
tensorrt fp32                6.803     6.803     6.839     1.32x     0.0000   1.16e-06  1.000000   100.0%
tensorrt fp16                1.757     1.738     1.825     5.09x     0.0536   3.50e-03  0.999993   100.0%


W1115 12:01:49.929000 402 torch/_inductor/compile_worker/subproc_pool.py:338] SubprocPool unclean exit


## 4. (선택) 빌드한 engine 공유하기
다음 study 때나 블로그 독자가 빌드를 건너뛸 수 있도록 GitHub Release에 올립니다.
1. 명령 팔레트 → `Colab: Download...`로 캐시 디렉토리의 `*.engine` 파일을 로컬 `artifacts/`에 받습니다.
2. 로컬에서 `scripts/publish_artifact.sh artifacts/*.engine`을 실행합니다.

engine은 **같은 GPU 아키텍처와 같은 TRT 버전**에서만 재사용할 수 있습니다(예: T4로 빌드한 engine은 L4에서 사용 불가).
이 노트북은 빌드 과정을 보여주려고 직접 빌드하지만, 이후 노트북에서는 `trt_utils.get_engine()`(캐시 → Release → 빌드)으로 받아 씁니다.